# QuickCart: Data Cleaning

This notebook prepares the raw quick-commerce dataset for SQL analysis, Power BI dashboarding, KPI monitoring, and decision intelligence development.

## Setup

In [38]:
import pandas as pd
import numpy as np

## Load Raw Data

In [39]:
df = pd.read_csv("quickcart_raw_orders.csv")
df.head()

,Order ID,Customer ID,Platform,Order Date & Time,Delivery Time (Minutes),Product Category,Order Value (INR),Customer Feedback,Service Rating,Delivery Delay,Refund Requested
0,ORD000001,CUST2824,JioMart,19:29.5,30,Fruits & Vegetables,382,"Fast delivery, great service!",5,No,No
1,ORD000002,CUST1409,Blinkit,54:29.5,16,Dairy,279,Quick and reliable!,5,No,No
2,ORD000003,CUST5506,JioMart,21:29.5,25,Beverages,599,Items missing from order.,2,No,Yes
3,ORD000004,CUST5012,JioMart,19:29.5,42,Beverages,946,Items missing from order.,2,Yes,Yes
4,ORD000005,CUST4657,Blinkit,49:29.5,30,Beverages,334,"Fast delivery, great service!",5,No,No


## Standardize Column Names

In [40]:
df.columns

Index(['Order ID', 'Customer ID', 'Platform', 'Order Date & Time',
       'Delivery Time (Minutes)', 'Product Category', 'Order Value (INR)',
       'Customer Feedback', 'Service Rating', 'Delivery Delay',
       'Refund Requested'],
      dtype='object')

In [41]:
df_clean = df.copy()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("&", "and")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)


In [42]:
df_clean.columns

Index(['order_id', 'customer_id', 'platform', 'order_date_and_time',
       'delivery_time_minutes', 'product_category', 'order_value_inr',
       'customer_feedback', 'service_rating', 'delivery_delay',
       'refund_requested'],
      dtype='object')

**Observation:** Column names were standardized to lowercase snake_case format for easier use in Python, SQL, and Power BI.

## Data Type Review

In [43]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   order_id               100000 non-null  object
 1   customer_id            100000 non-null  object
 2   platform               100000 non-null  object
 3   order_date_and_time    100000 non-null  object
 4   delivery_time_minutes  100000 non-null  int64 
 5   product_category       100000 non-null  object
 6   order_value_inr        100000 non-null  int64 
 7   customer_feedback      100000 non-null  object
 8   service_rating         100000 non-null  int64 
 9   delivery_delay         100000 non-null  object
 10  refund_requested       100000 non-null  object
dtypes: int64(3), object(8)
memory usage: 8.4+ MB


**Observation:** Numeric fields are correctly stored. The `order_date_and_time` field remains text-based and needs review before time-based analysis.

## Time Field Review

In [44]:
df_clean["order_date_and_time"].head(10)

,order_date_and_time
0,19:29.5
1,54:29.5
2,21:29.5
3,19:29.5
4,49:29.5
5,36:29.5
6,22:29.5
7,50:29.5
8,51:29.5
9,08:29.5


In [45]:
df_clean["order_date_and_time"].nunique()

60

In [46]:
time_format_check = df_clean["order_date_and_time"].astype(str).str.match(r"^\d{2}:\d{2}\.\d$")

time_format_check.value_counts()

,count
order_date_and_time,
True,100000


**Observation:** The time field follows a consistent pattern, but it does not represent a standard date-time format. It will be retained as a raw time-related field and excluded from time-based analysis unless a valid date-time structure is confirmed.

## Time Field Handling

In [47]:
df_clean = df_clean.rename(columns={
    "order_date_and_time": "order_time_raw"
})

df_clean[["order_time_raw"]].head()

,order_time_raw
0,19:29.5
1,54:29.5
2,21:29.5
3,19:29.5
4,49:29.5


In [48]:
time_pattern_match = df_clean["order_time_raw"].astype(str).str.match(r"^\d{2}:\d{2}\.\d$")

standard_clock_time_match = df_clean["order_time_raw"].astype(str).str.match(
    r"^(?:[01]\d|2[0-3]):[0-5]\d(?:\.\d)?$"
)

time_field_summary = pd.DataFrame({
    "check": [
        "Records following original pattern",
        "Records matching standard clock-time format",
        "Unique raw time values"
    ],
    "value": [
        time_pattern_match.sum(),
        standard_clock_time_match.sum(),
        df_clean["order_time_raw"].nunique()
    ]
})

time_field_summary

,check,value
0,Records following original pattern,100000
1,Records matching standard clock-time format,39911
2,Unique raw time values,60


**Observation:** The original time field was renamed to `order_time_raw` because it does not represent a reliable standard date-time value. The raw values are preserved, but this field will be excluded from time-based analysis unless its source meaning is confirmed.

## Categorical Value Validation

In [49]:
categorical_cols = [
    "platform",
    "product_category",
    "delivery_delay",
    "refund_requested"
]

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

categorical_summary = pd.DataFrame({
    "column_name": categorical_cols,
    "unique_values": [df_clean[col].nunique() for col in categorical_cols],
    "sample_values": [list(df_clean[col].unique()) for col in categorical_cols]
})

categorical_summary

,column_name,unique_values,sample_values
0,platform,3,"[JioMart, Blinkit, Swiggy Instamart]"
1,product_category,6,"[Fruits & Vegetables, Dairy, Beverages, Person..."
2,delivery_delay,2,"[No, Yes]"
3,refund_requested,2,"[No, Yes]"


**Observation:** Key categorical fields were validated and found to be consistent. Platform, product category, delivery delay, and refund request values are ready for analysis.

## Customer Feedback Classification

In [50]:
feedback_category_map = {
    "Fast delivery, great service!": "Positive Experience",
    "Quick and reliable!": "Positive Experience",
    "Excellent experience!": "Positive Experience",
    "Easy to order, loved it!": "Positive Experience",
    "Very satisfied with the service.": "Positive Experience",
    "Good quality products.": "Positive Experience",
    "Items missing from order.": "Order Issue",
    "Wrong item delivered.": "Order Issue",
    "Not fresh, disappointed.": "Product Quality Issue",
    "Packaging could be better.": "Packaging Issue",
    "Very late delivery, not happy.": "Delivery Issue",
    "Delivery person was rude.": "Service Issue",
    "Horrible experience, never ordering again.": "Negative Experience"
}

df_clean["feedback_category"] = df_clean["customer_feedback"].map(feedback_category_map)

df_clean[["customer_feedback", "feedback_category"]].head(10)

,customer_feedback,feedback_category
0,"Fast delivery, great service!",Positive Experience
1,Quick and reliable!,Positive Experience
2,Items missing from order.,Order Issue
3,Items missing from order.,Order Issue
4,"Fast delivery, great service!",Positive Experience
5,Items missing from order.,Order Issue
6,"Fast delivery, great service!",Positive Experience
7,"Horrible experience, never ordering again.",Negative Experience
8,Very satisfied with the service.,Positive Experience
9,"Very late delivery, not happy.",Delivery Issue


In [51]:
feedback_category_summary = df_clean["feedback_category"].value_counts().reset_index()
feedback_category_summary.columns = ["feedback_category", "order_count"]

feedback_category_summary["percentage"] = (
    feedback_category_summary["order_count"] / len(df_clean) * 100
).round(2)

feedback_category_summary

,feedback_category,order_count,percentage
0,Positive Experience,46477,46.48
1,Order Issue,15475,15.48
2,Packaging Issue,7704,7.70
3,Service Issue,7643,7.64
4,Delivery Issue,7592,7.59
5,Product Quality Issue,7580,7.58
6,Negative Experience,7529,7.53


In [52]:
df_clean["feedback_category"].isnull().sum()

np.int64(0)

**Observation:** Customer feedback was classified into business-friendly issue categories using rule-based mapping. All feedback values were successfully mapped, creating a new `feedback_category` column for customer experience, refund behavior, and decision intelligence analysis.

In [53]:
df_clean = df_clean.rename(columns={
    "customer_feedback": "customer_feedback_raw"
})

df_clean[["customer_feedback_raw", "feedback_category"]].head(10)

,customer_feedback_raw,feedback_category
0,"Fast delivery, great service!",Positive Experience
1,Quick and reliable!,Positive Experience
2,Items missing from order.,Order Issue
3,Items missing from order.,Order Issue
4,"Fast delivery, great service!",Positive Experience
5,Items missing from order.,Order Issue
6,"Fast delivery, great service!",Positive Experience
7,"Horrible experience, never ordering again.",Negative Experience
8,Very satisfied with the service.,Positive Experience
9,"Very late delivery, not happy.",Delivery Issue


**Observation:** The original feedback text was preserved as `customer_feedback_raw`, while `feedback_category` provides a cleaner business-ready classification for analysis.

## Business-Ready Feature Creation

In [54]:
df_clean["delay_flag"] = df_clean["delivery_delay"].map({
    "Yes": 1,
    "No": 0
})

df_clean["refund_flag"] = df_clean["refund_requested"].map({
    "Yes": 1,
    "No": 0
})

df_clean[["delivery_delay", "delay_flag", "refund_requested", "refund_flag"]].head()

,delivery_delay,delay_flag,refund_requested,refund_flag
0,No,0,No,0
1,No,0,No,0
2,No,0,Yes,1
3,Yes,1,Yes,1
4,No,0,No,0


In [55]:
def rating_group(rating):
    if rating >= 4:
        return "High Rating"
    elif rating == 3:
        return "Medium Rating"
    else:
        return "Low Rating"

df_clean["rating_group"] = df_clean["service_rating"].apply(rating_group)

df_clean[["service_rating", "rating_group"]].head()

,service_rating,rating_group
0,5,High Rating
1,5,High Rating
2,2,Low Rating
3,2,Low Rating
4,5,High Rating


In [56]:
delivery_p25 = df_clean["delivery_time_minutes"].quantile(0.25)
delivery_p75 = df_clean["delivery_time_minutes"].quantile(0.75)

def delivery_performance_group(minutes):
    if minutes <= delivery_p25:
        return "Fast Delivery"
    elif minutes <= delivery_p75:
        return "Standard Delivery"
    else:
        return "Slow Delivery"

df_clean["delivery_performance_group"] = df_clean["delivery_time_minutes"].apply(delivery_performance_group)

delivery_p25, delivery_p75

(np.float64(23.0), np.float64(36.0))

**Observation:** Delivery performance groups were created using dataset-based percentile thresholds. Orders at or below the 25th percentile were classified as Fast Delivery, orders up to the 75th percentile were classified as Standard Delivery, and orders above the 75th percentile were classified as Slow Delivery.

In [57]:
df_clean[["delivery_time_minutes", "delivery_performance_group"]].head()

,delivery_time_minutes,delivery_performance_group
0,30,Standard Delivery
1,16,Fast Delivery
2,25,Standard Delivery
3,42,Slow Delivery
4,30,Standard Delivery


In [58]:
df_clean["order_value_segment"] = pd.qcut(
    df_clean["order_value_inr"],
    q=3,
    labels=["Low Value Order", "Medium Value Order", "High Value Order"]
)

order_value_segment_summary = (
    df_clean
    .groupby("order_value_segment", observed=True)
    .agg(
        min_order_value=("order_value_inr", "min"),
        max_order_value=("order_value_inr", "max"),
        order_count=("order_id", "count")
    )
    .reset_index()
)

order_value_segment_summary["order_percentage"] = (
    order_value_segment_summary["order_count"] / len(df_clean) * 100
).round(2)

order_value_segment_summary

,order_value_segment,min_order_value,max_order_value,order_count,order_percentage
0,Low Value Order,50,349,33416,33.42
1,Medium Value Order,350,659,33338,33.34
2,High Value Order,660,2000,33246,33.25


In [59]:
df_clean[["order_value_inr", "order_value_segment"]].head(5)

,order_value_inr,order_value_segment
0,382,Medium Value Order
1,279,Low Value Order
2,599,Medium Value Order
3,946,High Value Order
4,334,Low Value Order


In [60]:
new_feature_cols = [
    "delay_flag",
    "refund_flag",
    "rating_group",
    "delivery_performance_group",
    "order_value_segment",
    "feedback_category"
]

df_clean[new_feature_cols].head()

,delay_flag,refund_flag,rating_group,delivery_performance_group,order_value_segment,feedback_category
0,0,0,High Rating,Standard Delivery,Medium Value Order,Positive Experience
1,0,0,High Rating,Fast Delivery,Low Value Order,Positive Experience
2,0,1,Low Rating,Standard Delivery,Medium Value Order,Order Issue
3,1,1,Low Rating,Slow Delivery,High Value Order,Order Issue
4,0,0,High Rating,Standard Delivery,Low Value Order,Positive Experience


**Observation:** Business-ready features were created for delay tracking, refund tracking, rating analysis, delivery performance grouping, order value segmentation, and feedback classification. Delivery and order value segments were created using dataset-based thresholds, making the features more reliable for SQL, Power BI, KPI monitoring, and decision intelligence analysis.

In [61]:
feature_validation = pd.DataFrame({
    "feature_name": new_feature_cols,
    "missing_values": [df_clean[col].isnull().sum() for col in new_feature_cols],
    "unique_values": [df_clean[col].nunique() for col in new_feature_cols],
    "sample_values": [list(df_clean[col].dropna().unique()[:5]) for col in new_feature_cols]
})

feature_validation

,feature_name,missing_values,unique_values,sample_values
0,delay_flag,0,2,"[0, 1]"
1,refund_flag,0,2,"[0, 1]"
2,rating_group,0,3,"[High Rating, Low Rating, Medium Rating]"
3,delivery_performance_group,0,3,"[Standard Delivery, Fast Delivery, Slow Delivery]"
4,order_value_segment,0,3,"[Medium Value Order, Low Value Order, High Val..."
5,feedback_category,0,7,"[Positive Experience, Order Issue, Negative Ex..."


**Observation:** All newly created business-ready features were successfully generated with no missing values.

## Final Data Quality Check

In [62]:
final_quality_check = pd.DataFrame({
    "check": [
        "Total rows",
        "Total columns",
        "Missing values",
        "Duplicate rows",
        "Unique order IDs",
        "Unique customers"
    ],
    "value": [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum(),
        df_clean["order_id"].nunique(),
        df_clean["customer_id"].nunique()
    ]
})

final_quality_check

,check,value
0,Total rows,100000
1,Total columns,17
2,Missing values,0
3,Duplicate rows,0
4,Unique order IDs,100000
5,Unique customers,9000


**Observation:** The cleaned dataset passed the final quality check and is ready for export.

## Export Cleaned Dataset

In [63]:
df_clean.to_csv("quickcart_cleaned_orders.csv", index=False)

In [64]:
export_check = pd.read_csv("quickcart_cleaned_orders.csv")

export_check.head()

,order_id,customer_id,platform,order_time_raw,delivery_time_minutes,product_category,order_value_inr,customer_feedback_raw,service_rating,delivery_delay,refund_requested,feedback_category,delay_flag,refund_flag,rating_group,delivery_performance_group,order_value_segment
0,ORD000001,CUST2824,JioMart,19:29.5,30,Fruits & Vegetables,382,"Fast delivery, great service!",5,No,No,Positive Experience,0,0,High Rating,Standard Delivery,Medium Value Order
1,ORD000002,CUST1409,Blinkit,54:29.5,16,Dairy,279,Quick and reliable!,5,No,No,Positive Experience,0,0,High Rating,Fast Delivery,Low Value Order
2,ORD000003,CUST5506,JioMart,21:29.5,25,Beverages,599,Items missing from order.,2,No,Yes,Order Issue,0,1,Low Rating,Standard Delivery,Medium Value Order
3,ORD000004,CUST5012,JioMart,19:29.5,42,Beverages,946,Items missing from order.,2,Yes,Yes,Order Issue,1,1,Low Rating,Slow Delivery,High Value Order
4,ORD000005,CUST4657,Blinkit,49:29.5,30,Beverages,334,"Fast delivery, great service!",5,No,No,Positive Experience,0,0,High Rating,Standard Delivery,Low Value Order


In [65]:
export_check.shape

(100000, 17)

**Observation:** The cleaned dataset was exported successfully as `quickcart_cleaned_orders.csv` and is ready for SQL analysis, Power BI dashboarding, KPI monitoring, and decision intelligence development.

## Cleaning Completion Checklist

| Status | Cleaning Action                  | Output                                                                              |
| ------ | -------------------------------- | ----------------------------------------------------------------------------------- |
| ✅      | Standardized column names        | Lowercase `snake_case` column names                                                 |
| ✅      | Reviewed data types              | Numeric fields confirmed as correctly stored                                        |
| ✅      | Handled non-standard time field  | Renamed to `order_time_raw` and preserved as raw reference                          |
| ✅      | Validated categorical fields     | Platform, product category, delay, and refund values confirmed as consistent        |
| ✅      | Preserved original feedback text | Renamed to `customer_feedback_raw`                                                  |
| ✅      | Created feedback classification  | Added `feedback_category` using business-rule-based mapping                         |
| ✅      | Created business-ready features  | Added delay flags, refund flags, rating groups, delivery performance groups, and order value segments             |
| ✅      | Performed final quality check    | Cleaned dataset verified before export                                              |
| ✅      | Exported cleaned dataset         | Dataset ready for SQL, Power BI, KPI monitoring, and decision intelligence analysis |

**Final Output:** The cleaned QuickCart dataset is ready for the next phase of analysis.
